# TitanicLogic nằm trong repo [AI_projects](https://github.com/DoanCongPho/AI_projects), thư mục `src/`.Chạy lại cell "Đồng bộ code" bên dưới để lấy bản mới nhất mà không cần upload gì.Cần bật **Internet** trong Settings.

## Đồng bộ code từ GitHub

In [ ]:
REPO = "https://github.com/DoanCongPho/AI_projects.git"
CODE = "/kaggle/working/code"

import os, sys, subprocess

if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "--depth", "1", REPO, CODE], check=True)
else:
    subprocess.run(["git", "-C", CODE, "pull", "--ff-only"], check=True)

if CODE not in sys.path:
    sys.path.insert(0, CODE)

from src.bootstrap import reload_src
reload_src()
print("code đã đồng bộ")

## Dữ liệu

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import pandas as pd
from src.titanic import load, add_features, FEATURES

train_raw, test_raw = load()
train = add_features(train_raw)
test = add_features(test_raw)

train.info()

## Khám pháBa nhóm dưới đây cho thấy `Sex` là tín hiệu mạnh nhất, sau đó tới `Pclass`.

In [ ]:
train.groupby('Sex')['Survived'].mean()

In [ ]:
train.groupby('Pclass')['Survived'].mean()

In [ ]:
train.groupby('Embarked')['Survived'].mean()

In [ ]:
# Danh xưng trích từ Name: Master là trẻ em nam, tách bạch hẳn khỏi Mr.
train.groupby('Title')['Survived'].agg(['mean', 'count'])

## Mô hìnhToàn bộ tiền xử lý nằm trong Pipeline nên OneHotEncoder chỉ học trên phầntrain của từng fold, tránh rò rỉ dữ liệu sang phần validation.

In [ ]:
from src.model import build_pipeline, evaluate

pipe = build_pipeline()
scores = evaluate(pipe, train, folds=5)

print("accuracy từng fold:", scores.round(4))
print("trung bình: %.4f (±%.4f)" % (scores.mean(), scores.std()))

## Nộp bài

In [ ]:
from src.model import make_submission

sub = make_submission(pipe, train, test, path="submission.csv")
print(sub.shape)
sub.head()